# Score-Based Models, SDEs, and Probability Flow ODEs

The discrete diffusion picture can be pushed further by asking what happens when the time step becomes very small and the number of steps becomes very large. In that regime, the noising chain is better described by a **stochastic differential equation**, and the reverse dynamics become a continuous-time denoising flow. This viewpoint does not replace DDPMs. It clarifies what the denoiser has really learned and why the same trained network can support several different samplers.

At the center of the discussion is the **score field** $\nabla_{\boldsymbol{x}} \log p_t(\boldsymbol{x})$. In the discrete chapter the model was trained to predict the Gaussian noise added at each step. Here the same object is reinterpreted geometrically: the denoiser is learning, at every noise level, how probability mass bends back toward the data distribution. Once that field is known, one can generate samples either with a stochastic reverse-time SDE or with a deterministic probability-flow ODE.

The continuous-time picture becomes especially useful when it is read alongside the DDPM implementation. The same trained FashionMNIST denoiser can be turned into a **score network** with one explicit formula. After that, generation becomes a question of choosing a solver and understanding what kind of trajectory that solver follows. In this sense, the chapter is less about inventing a new model family than about revealing the structure that was already present inside the DDPM.

```{important}
A pretrained **noise predictor** already contains a **score model**. Continuous-time diffusion is the language that makes this statement precise.
```

## Deterministic Flows, Stochastic Flows, and Scores

An **ordinary differential equation** describes deterministic motion:
:::{math}
\frac{d\boldsymbol{x}}{dt} = \boldsymbol{f}(\boldsymbol{x}(t), t).
:::
Once the initial condition is fixed, the entire trajectory is fixed. A **stochastic differential equation** adds continuous random perturbations:
:::{math}
d\boldsymbol{x} = \boldsymbol{f}(\boldsymbol{x}, t)\,dt + g(t)\,d\boldsymbol{w},
:::
where $\boldsymbol{w}$ is Brownian motion. The drift $\boldsymbol{f}$ pushes the sample systematically, while the diffusion term keeps scattering it. This is the right language for forward noising, because a good corruption process should not merely deform the data manifold; it should actually spread probability mass until the distribution becomes simple enough to sample from directly.

The reverse problem is therefore a problem of geometry. If a noisy sample drifts away from the high-probability region, what vector field should bring it back? The answer is the **score**. If $p_t$ denotes the marginal distribution at time $t$, then
:::{math}
\nabla_{\boldsymbol{x}} \log p_t(\boldsymbol{x})
:::
points locally toward directions where the density increases. This is why score estimation and denoising are so tightly connected: a good score field tells the reverse dynamics how to pull a noisy configuration back toward more plausible image structure.

```{admonition} Numerical Example: A One-Dimensional Score
:class: numerical-example
Suppose that at some noise level the marginal density were a Gaussian centered at $2$ with unit variance. Then
:::{math}
\nabla_x \log p_t(x) = -(x-2).
:::
If the current sample is $x=5$, the score is $-3$, so the vector field points left, toward the high-density region. If the current sample is $x=0$, the score is $2$, so the vector field points right. The score does not report the density value itself. It reports the **local direction in which the sample should move to become more probable**.
```

## Denoising Score Matching

Consider Gaussian perturbations
:::{math}
\widetilde{\boldsymbol{x}} = \boldsymbol{x} + \sigma \boldsymbol{\varepsilon},
\qquad
\boldsymbol{\varepsilon} \sim \mathcal{N}(\boldsymbol{0},\boldsymbol{I}),
:::
with $\boldsymbol{x} \sim p_{gt}$. Let $p_\sigma$ be the density of the perturbed variable $\widetilde{\boldsymbol{x}}$. A network can be trained to estimate the score of that noisy density by minimizing a regression loss against the known Gaussian conditional score
:::{math}
\nabla_{\widetilde{\boldsymbol{x}}}\log q_\sigma(\widetilde{\boldsymbol{x}}|\boldsymbol{x})
=
-\frac{\widetilde{\boldsymbol{x}}-\boldsymbol{x}}{\sigma^2}.
:::
This is the core simplification. A quantity that looks intractable at the level of the unknown noisy data density becomes a plain supervised target once the corruption mechanism is explicit.

```{prf:theorem} Denoising score matching recovers the noisy-data score
:label: thm-dsm-fashion
The minimizer of
:::{math}
\mathcal{L}(\theta)
=
\mathbb{E}\left[
\left\|
\boldsymbol{s}_\theta(\widetilde{\boldsymbol{x}},\sigma)
+
\frac{\widetilde{\boldsymbol{x}}-\boldsymbol{x}}{\sigma^2}
\right\|_2^2
\right]
:::
is the score of the perturbed density:
:::{math}
\boldsymbol{s}_\theta^*(\widetilde{\boldsymbol{x}},\sigma)
=
\nabla_{\widetilde{\boldsymbol{x}}}\log p_\sigma(\widetilde{\boldsymbol{x}}).
:::
```

```{prf:proof}
Condition on $\widetilde{\boldsymbol{x}}$. The optimal regression target is the conditional expectation of the known Gaussian target:
:::{math}
\boldsymbol{s}_\theta^*(\widetilde{\boldsymbol{x}},\sigma)
=
\mathbb{E}\left[
-\frac{\widetilde{\boldsymbol{x}}-\boldsymbol{x}}{\sigma^2}
\middle|
\widetilde{\boldsymbol{x}}
\right].
:::
The denoising-score-matching identity shows that this conditional expectation is exactly the score of the perturbed density,
:::{math}
\nabla_{\widetilde{\boldsymbol{x}}}\log p_\sigma(\widetilde{\boldsymbol{x}}).
:::
The network is therefore optimized against a target induced by the known corruption law, but the optimum is the score of the unknown noisy-data distribution.
```

This is the continuous-time analogue of what happened in DDPM training. The generative problem is not attacked by trying to estimate a high-dimensional density directly. It is attacked by converting the problem into a regression target produced by Gaussian corruption. The score-based view and the DDPM view are already very close here: both rely on the fact that the perturbation process is explicit enough to manufacture a supervised signal.

## The Variance-Preserving SDE and the Continuous Limit of DDPMs

The main continuous-time family connected with DDPMs is the **variance-preserving SDE**
:::{math}
d\boldsymbol{x}
=
-\frac{1}{2}\beta(t)\boldsymbol{x}\,dt + \sqrt{\beta(t)}\,d\boldsymbol{w}.
:::
Its drift gradually damps the signal, while its diffusion term injects Gaussian noise. If one defines the continuous cumulative retention factor
:::{math}
\bar{\alpha}(t)
=
\exp\left(-\int_0^t \beta(s)\,ds\right),
:::
then the forward marginal is
:::{math}
\boldsymbol{x}_t
=
\sqrt{\bar{\alpha}(t)}\,\boldsymbol{x}_0 + \sqrt{1-\bar{\alpha}(t)}\,\boldsymbol{\varepsilon},
\qquad
\boldsymbol{\varepsilon}\sim\mathcal{N}(\boldsymbol{0},\boldsymbol{I}).
:::
This is exactly the same algebraic structure as the discrete DDPM forward marginal, except that the cumulative signal-retention coefficient is now written as a smooth function of continuous time.

```{prf:theorem} VP-SDE forward marginal
:label: thm-vp-marginal-fashion
For the variance-preserving SDE,
:::{math}
d\boldsymbol{x}
=
-\frac{1}{2}\beta(t)\boldsymbol{x}\,dt + \sqrt{\beta(t)}\,d\boldsymbol{w},
:::
the conditional law of $\boldsymbol{x}_t$ given $\boldsymbol{x}_0$ is Gaussian with
:::{math}
p(\boldsymbol{x}_t|\boldsymbol{x}_0)
=
\mathcal{N}\left(
\sqrt{\bar{\alpha}(t)}\,\boldsymbol{x}_0,
(1-\bar{\alpha}(t))\boldsymbol{I}
\right).
:::
```

```{prf:theorem} DDPM as a discretized VP diffusion
:label: thm-ddpm-is-vp
If continuous time is discretized into finitely many noise levels, the VP-SDE forward marginal reduces to the DDPM forward marginal with cumulative coefficients $\alpha_t = \prod_{s=1}^t \bar{\alpha}_s$. The DDPM reverse chain is therefore a finite-step approximation of the reverse-time VP diffusion.
```

```{prf:proof}
In the continuous model the marginal takes the form
:::{math}
\boldsymbol{x}_t = \sqrt{\bar{\alpha}(t)}\,\boldsymbol{x}_0 + \sqrt{1-\bar{\alpha}(t)}\,\boldsymbol{\varepsilon}.
:::
Sampling the time axis at discrete points $t_1,\dots,t_T$ gives a sequence of cumulative retention factors. Writing these sampled factors as $\alpha_t$ yields exactly the DDPM marginal
:::{math}
\boldsymbol{x}_t = \sqrt{\alpha_t}\,\boldsymbol{x}_0 + \sqrt{1-\alpha_t}\,\boldsymbol{\varepsilon}.
:::
The corresponding reverse-time dynamics, when approximated by Gaussian one-step transitions, produce the DDPM reverse sampler. The discrete chain is therefore the time-discretized version of the same variance-preserving diffusion.
```

## From a Noise Predictor to a Score Network

Now the key identity can be stated. Suppose the denoiser is trained in the DDPM way, so that
:::{math}
\boldsymbol{\varepsilon}_\theta(\boldsymbol{x}_t,t) \approx \boldsymbol{\varepsilon}.
:::
For the VP perturbation path, the Gaussian conditional score is explicit. Substituting the forward reparameterization into that score gives a direct conversion formula from predicted noise to score.

```{prf:theorem} Score from a pretrained noise predictor
:label: thm-score-from-noise-fashion
For the variance-preserving forward marginal,
:::{math}
\boldsymbol{s}_\theta(\boldsymbol{x}_t,t)
=
-\frac{\boldsymbol{\varepsilon}_\theta(\boldsymbol{x}_t,t)}{\sqrt{1-\bar{\alpha}(t)}}.
:::
In the discrete DDPM notation this becomes
:::{math}
\boldsymbol{s}_\theta(\boldsymbol{x}_t,t)
=
-\frac{\boldsymbol{\varepsilon}_\theta(\boldsymbol{x}_t,t)}{\sqrt{1-\alpha_t}}.
:::
```

```{prf:proof}
The Gaussian forward conditional satisfies
:::{math}
\nabla_{\boldsymbol{x}_t}\log p(\boldsymbol{x}_t|\boldsymbol{x}_0)
=
-\frac{\boldsymbol{x}_t-\sqrt{\bar{\alpha}(t)}\,\boldsymbol{x}_0}{1-\bar{\alpha}(t)}.
:::
Using
:::{math}
\boldsymbol{x}_t-\sqrt{\bar{\alpha}(t)}\,\boldsymbol{x}_0
=
\sqrt{1-\bar{\alpha}(t)}\,\boldsymbol{\varepsilon},
:::
one gets
:::{math}
\nabla_{\boldsymbol{x}_t}\log p(\boldsymbol{x}_t|\boldsymbol{x}_0)
=
-\frac{\boldsymbol{\varepsilon}}{\sqrt{1-\bar{\alpha}(t)}}.
:::
Replacing the unknown noise by the neural estimate gives the claimed score formula.
```

This identity is the exact bridge from the previous notebook to the current one. Once a DDPM denoiser has been trained, there is no second conceptual leap to score modeling. The score field is already there, encoded in the noise predictor. What changes afterward is only the **sampling rule**: one can either keep the stochastic reverse diffusion or integrate a deterministic flow with the same marginals.

## Reverse-Time SDE, Probability-Flow ODE, and DDIM

The reverse-time SDE for a generic forward diffusion
:::{math}
d\boldsymbol{x} = \boldsymbol{f}(\boldsymbol{x},t)\,dt + g(t)\,d\boldsymbol{w}
:::
has drift
:::{math}
\boldsymbol{f}(\boldsymbol{x},t) - g(t)^2\nabla_{\boldsymbol{x}}\log p_t(\boldsymbol{x}).
:::
For the variance-preserving family this becomes
:::{math}
d\boldsymbol{x}
=
\left[
-\frac{1}{2}\beta(t)\boldsymbol{x} - \beta(t)\nabla_{\boldsymbol{x}}\log p_t(\boldsymbol{x})
\right]dt + \sqrt{\beta(t)}\,d\bar{\boldsymbol{w}}.
:::
The score term is what turns a forward noising law into a reverse denoising law. If the score is known, one can simulate this reverse SDE directly.

The **probability-flow ODE** replaces the stochastic update with a deterministic one that preserves the same one-time marginals:
:::{math}
\frac{d\boldsymbol{x}}{dt}
=
\boldsymbol{f}(\boldsymbol{x},t) - \frac{1}{2}g(t)^2\nabla_{\boldsymbol{x}}\log p_t(\boldsymbol{x}).
:::
The field is almost the same. The stochastic term disappears, and the score correction is halved. This is the continuous-time reason why one trained denoiser can support both stochastic and deterministic sampling.

```{prf:theorem} DDIM as a deterministic discretization of probability-flow sampling
:label: thm-ddim-pfode-fashion
For a variance-preserving diffusion, the DDIM update with $\eta = 0$ is a deterministic discretization of the same score-driven denoising geometry captured by the probability-flow ODE. Once the terminal noise sample is fixed, the DDIM trajectory is fixed.
```

```{prf:proof}
DDIM with $\eta=0$ first reconstructs
:::{math}
\hat{\boldsymbol{x}}_{0|t}
=
\frac{\boldsymbol{x}_t - \sqrt{1-\alpha_t}\,\boldsymbol{\varepsilon}_\theta(\boldsymbol{x}_t,t)}{\sqrt{\alpha_t}},
:::
and then updates deterministically via
:::{math}
\boldsymbol{x}_{t'}
=
\sqrt{\alpha_{t'}}\,\hat{\boldsymbol{x}}_{0|t}
+
\sqrt{1-\alpha_{t'}}\,\boldsymbol{\varepsilon}_\theta(\boldsymbol{x}_t,t).
:::
No fresh Gaussian perturbation is added, so once $\boldsymbol{x}_T$ is fixed the entire path is fixed. In the small-step limit this is exactly the kind of deterministic score-driven transport described by the probability-flow ODE, written in the $\hat{\boldsymbol{x}}_{0|t}$ and noise-prediction parameterization rather than directly in terms of the score.
```

At this point the geometry of the diffusion family is visible in a very compact way. **DDPM ancestral sampling** is a stochastic discretization of the reverse-time dynamics. **DDIM** is a deterministic discretization of the same score-induced denoising field. The continuous-time formulation does not create a second family next to DDPM. It reveals the common structure behind these samplers.

## Using the FashionMNIST DDPM Denoiser as a Score Network

The implementation below uses the same `FashionMNIST` denoiser trained in the DDPM notebook. The network architecture, time embedding, and discrete schedule are kept identical. After loading the weights, the score is obtained through the theorem above, and the resulting field is used to run three samplers on the same model:

- the **ancestral DDPM sampler**;
- an **Euler-Maruyama reverse-SDE sampler**;
- a **probability-flow ODE sampler**.

Because all three samplers are driven by the same denoiser, the comparison is about the geometry of the reverse dynamics rather than about differences in training.

### Loading the Pretrained DDPM Denoiser

In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance
from torchvision import datasets, transforms, utils
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(7)
if device.type == "cuda":
    torch.cuda.manual_seed_all(7)

project_root = Path.cwd() if (Path.cwd() / "_config.yml").exists() else Path.cwd().parent
artifacts_dir = project_root / "artifacts"
artifacts_dir.mkdir(exist_ok=True)
ddpm_checkpoint_path = artifacts_dir / "ddpm_fashionmnist.pt"

batch_size = 128
metric_batch_size = 64
num_workers = 2 if device.type == "cuda" else 0
image_size = 28
channels = 1
base_channels = 64
time_dim = 128
timesteps = 300

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: 2.0 * x - 1.0),
])

test_dataset = datasets.FashionMNIST(root=project_root / "data", train=False, download=True, transform=transform)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=(device.type == "cuda"),
)


In [ ]:
betas = torch.linspace(1e-4, 0.02, timesteps, device=device)
bar_alphas = 1.0 - betas
alphas = torch.cumprod(bar_alphas, dim=0)
alphas_prev = torch.cat([torch.tensor([1.0], device=device), alphas[:-1]], dim=0)

sqrt_alphas = torch.sqrt(alphas)
sqrt_one_minus_alphas = torch.sqrt(1.0 - alphas)
sqrt_recip_bar_alphas = torch.sqrt(1.0 / bar_alphas)
posterior_variance = betas * (1.0 - alphas_prev) / (1.0 - alphas)


def extract(coefficients, t, x_shape):
    out = coefficients.gather(0, t)
    return out.view(t.shape[0], *((1,) * (len(x_shape) - 1)))


In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half_dim = self.dim // 2
        factor = math.log(10000.0) / max(half_dim - 1, 1)
        frequencies = torch.exp(torch.arange(half_dim, device=t.device) * -factor)
        angles = t.float().unsqueeze(1) * frequencies.unsqueeze(0)
        embedding = torch.cat([angles.sin(), angles.cos()], dim=1)
        if self.dim % 2 == 1:
            embedding = F.pad(embedding, (0, 1))
        return embedding


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.norm2 = nn.GroupNorm(8, out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.time_proj = nn.Linear(time_dim, out_channels)
        self.skip = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()

    def forward(self, x, time_emb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.time_proj(time_emb)[:, :, None, None]
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)


class SmallUNet(nn.Module):
    def __init__(self, in_channels=1, base_channels=64, time_dim=128):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )
        self.input_conv = nn.Conv2d(in_channels, base_channels, kernel_size=3, padding=1)
        self.down1 = ResidualBlock(base_channels, base_channels, time_dim)
        self.downsample = nn.Conv2d(base_channels, base_channels * 2, kernel_size=4, stride=2, padding=1)
        self.mid1 = ResidualBlock(base_channels * 2, base_channels * 2, time_dim)
        self.mid2 = ResidualBlock(base_channels * 2, base_channels * 2, time_dim)
        self.upsample = nn.ConvTranspose2d(base_channels * 2, base_channels, kernel_size=4, stride=2, padding=1)
        self.up1 = ResidualBlock(base_channels * 2, base_channels, time_dim)
        self.output_conv = nn.Conv2d(base_channels, in_channels, kernel_size=3, padding=1)

    def forward(self, x, t):
        time_emb = self.time_mlp(t)
        x0 = self.input_conv(x)
        x1 = self.down1(x0, time_emb)
        x2 = self.downsample(x1)
        x2 = self.mid1(x2, time_emb)
        x2 = self.mid2(x2, time_emb)
        x3 = self.upsample(x2)
        x3 = torch.cat([x3, x1], dim=1)
        x3 = self.up1(x3, time_emb)
        return self.output_conv(x3)


model = SmallUNet(in_channels=channels, base_channels=base_channels, time_dim=time_dim).to(device)
if not ddpm_checkpoint_path.exists():
    raise FileNotFoundError(f"Missing DDPM weights at {ddpm_checkpoint_path}. Run the discrete diffusion notebook first.")
model.load_state_dict(torch.load(ddpm_checkpoint_path, map_location=device))
model.eval()


### Reconstructing $\hat{\boldsymbol{x}}_{0|t}$ and the Score Field

The easiest place to see the score interpretation in action is to start from a real image, corrupt it at a chosen time, and then reconstruct the clean image estimate
:::{math}
\hat{\boldsymbol{x}}_{0|t}
=
\frac{\boldsymbol{x}_t - \sqrt{1-\alpha_t}\,\boldsymbol{\varepsilon}_\theta(\boldsymbol{x}_t,t)}{\sqrt{\alpha_t}}.
:::
This is the same estimate that appeared in DDIM. The score formula says that the corresponding score is simply the predicted noise rescaled by the current noise standard deviation.

In [ ]:
@torch.no_grad()
def q_sample(x0, t, noise=None):
    if noise is None:
        noise = torch.randn_like(x0)
    return (
        extract(sqrt_alphas, t, x0.shape) * x0
        + extract(sqrt_one_minus_alphas, t, x0.shape) * noise
    )


@torch.no_grad()
def estimate_x0_and_score(model, xt, t):
    pred_noise = model(xt, t)
    x0_hat = (xt - extract(sqrt_one_minus_alphas, t, xt.shape) * pred_noise) / extract(sqrt_alphas, t, xt.shape)
    score = -pred_noise / extract(sqrt_one_minus_alphas, t, xt.shape).clamp(min=1e-5)
    return x0_hat, score


In [ ]:
x0, _ = next(iter(test_loader))
x0 = x0[:8].to(device)
selected_steps = torch.tensor([10, 60, 120, 200, 280, 10, 120, 280], device=device)
xt = q_sample(x0, selected_steps)
x0_hat, score = estimate_x0_and_score(model, xt, selected_steps)

visual = torch.cat([
    0.5 * (x0 + 1.0),
    0.5 * (xt.clamp(-1, 1) + 1.0),
    0.5 * (x0_hat.clamp(-1, 1) + 1.0),
], dim=0)

plt.figure(figsize=(14, 5))
plt.imshow(utils.make_grid(visual.cpu(), nrow=8, pad_value=1.0).permute(1, 2, 0), cmap='gray')
plt.axis('off')
plt.title('Top: real images | middle: noisy images | bottom: denoised estimates')
plt.show()


### Three Samplers Driven by the Same Denoiser

The ancestral DDPM sampler, the reverse-time SDE solver, and the probability-flow ODE solver all start from Gaussian noise and query the same network. They differ only in how they convert the score information into a trajectory.

The reverse-time SDE solver below uses **Euler-Maruyama**. The probability-flow ODE solver uses **Euler integration**. To keep the link with the trained DDPM denoiser explicit, the continuous-time update uses a time grid in $[1,0]$ but evaluates the network at the nearest discrete DDPM step.

In [ ]:
@torch.no_grad()
def sample_ddpm(model, n_samples=16, show_progress=True):
    model.eval()
    x = torch.randn(n_samples, channels, image_size, image_size, device=device)

    reverse_steps = reversed(range(timesteps))
    if show_progress:
        reverse_steps = tqdm(reverse_steps, total=timesteps, desc="ddpm sampling", leave=False)
    for t_scalar in reverse_steps:
        t = torch.full((n_samples,), t_scalar, device=device, dtype=torch.long)
        predicted_noise = model(x, t)
        beta_t = extract(betas, t, x.shape)
        sqrt_one_minus_alpha_t = extract(sqrt_one_minus_alphas, t, x.shape)
        sqrt_recip_bar_alpha_t = extract(sqrt_recip_bar_alphas, t, x.shape)
        model_mean = sqrt_recip_bar_alpha_t * (x - beta_t * predicted_noise / sqrt_one_minus_alpha_t)

        if t_scalar > 0:
            variance = extract(posterior_variance, t, x.shape)
            x = model_mean + torch.sqrt(variance) * torch.randn_like(x)
        else:
            x = model_mean

    x = x.clamp(-1, 1)
    return 0.5 * (x + 1.0)


@torch.no_grad()
def sample_reverse_sde(model, n_samples=16, num_steps=300, show_progress=True, x_init=None):
    model.eval()
    x = torch.randn(n_samples, channels, image_size, image_size, device=device) if x_init is None else x_init.clone().to(device)
    time_grid = torch.linspace(1.0, 1.0 / timesteps, num_steps, device=device)
    dt = -(time_grid[0] - time_grid[1]).item()

    iterator = time_grid
    if show_progress:
        iterator = tqdm(time_grid, total=num_steps, desc="reverse sde", leave=False)

    for tau in iterator:
        t_index = torch.full((n_samples,), min(int(round(tau.item() * (timesteps - 1))), timesteps - 1), device=device, dtype=torch.long)
        beta_tau = betas[t_index].view(-1, 1, 1, 1)
        pred_noise = model(x, t_index)
        score = -pred_noise / extract(sqrt_one_minus_alphas, t_index, x.shape).clamp(min=1e-5)
        drift = -0.5 * beta_tau * x - beta_tau * score
        x = x + drift * dt + torch.sqrt(beta_tau * abs(dt)) * torch.randn_like(x)

    x = x.clamp(-1, 1)
    return 0.5 * (x + 1.0)


@torch.no_grad()
def sample_probability_flow_ode(model, n_samples=16, num_steps=300, show_progress=True, x_init=None):
    model.eval()
    x = torch.randn(n_samples, channels, image_size, image_size, device=device) if x_init is None else x_init.clone().to(device)
    time_grid = torch.linspace(1.0, 1.0 / timesteps, num_steps, device=device)
    dt = -(time_grid[0] - time_grid[1]).item()

    iterator = time_grid
    if show_progress:
        iterator = tqdm(time_grid, total=num_steps, desc="pf ode", leave=False)

    for tau in iterator:
        t_index = torch.full((n_samples,), min(int(round(tau.item() * (timesteps - 1))), timesteps - 1), device=device, dtype=torch.long)
        beta_tau = betas[t_index].view(-1, 1, 1, 1)
        pred_noise = model(x, t_index)
        score = -pred_noise / extract(sqrt_one_minus_alphas, t_index, x.shape).clamp(min=1e-5)
        velocity = -0.5 * beta_tau * x - 0.5 * beta_tau * score
        x = x + velocity * dt

    x = x.clamp(-1, 1)
    return 0.5 * (x + 1.0)


### Visual Comparison of DDPM, Reverse SDE, and Probability-Flow ODE

Looking at the generated grids makes the distinction concrete. The ancestral DDPM and the reverse-time SDE both keep randomness in the trajectory. The probability-flow ODE produces a deterministic transport once the terminal Gaussian sample is fixed.

In [ ]:
ddpm_samples = sample_ddpm(model, n_samples=16)
sde_samples = sample_reverse_sde(model, n_samples=16, num_steps=300)
pf_samples = sample_probability_flow_ode(model, n_samples=16, num_steps=300)

for title, samples in [
    ("Ancestral DDPM samples", ddpm_samples),
    ("Reverse-time SDE samples", sde_samples),
    ("Probability-flow ODE samples", pf_samples),
]:
    plt.figure(figsize=(6, 6))
    plt.imshow(utils.make_grid(samples.cpu(), nrow=4, pad_value=1.0).permute(1, 2, 0), cmap='gray')
    plt.axis('off')
    plt.title(title)
    plt.show()


### Fixing the Terminal Noise

A deterministic solver reveals itself most clearly when the starting noise is held fixed. Re-running the probability-flow ODE from the same terminal sample produces the same trajectory. Re-running the reverse-time SDE does not, because fresh Gaussian perturbations are injected during integration.

In [ ]:
shared_noise = torch.randn(8, channels, image_size, image_size, device=device)

pf_first = sample_probability_flow_ode(model, n_samples=8, num_steps=300, x_init=shared_noise)
pf_second = sample_probability_flow_ode(model, n_samples=8, num_steps=300, x_init=shared_noise)

sde_first = sample_reverse_sde(model, n_samples=8, num_steps=300, x_init=shared_noise)
sde_second = sample_reverse_sde(model, n_samples=8, num_steps=300, x_init=shared_noise)

for title, samples in [
    ("PF-ODE from fixed terminal noise | run 1", pf_first),
    ("PF-ODE from fixed terminal noise | run 2", pf_second),
    ("Reverse SDE from fixed terminal noise | run 1", sde_first),
    ("Reverse SDE from fixed terminal noise | run 2", sde_second),
]:
    plt.figure(figsize=(10, 2.6))
    plt.imshow(utils.make_grid(samples.cpu(), nrow=8, pad_value=1.0).permute(1, 2, 0), cmap='gray')
    plt.axis('off')
    plt.title(title)
    plt.show()


### Batched FID and KID Across Samplers

Because the denoiser is fixed, the metric comparison is genuinely a comparison among samplers. The real-image statistics are accumulated once, while fake images are generated in small batches to keep memory usage controlled.

In [ ]:
def prepare_for_metrics(images):
    if images.size(1) == 1:
        images = images.repeat(1, 3, 1, 1)
    return images.clamp(0.0, 1.0)


@torch.no_grad()
def compute_sampler_metrics(sample_fn, real_loader, num_fake=1000, metric_batch_size=64):
    fid = FrechetInceptionDistance(feature=2048, normalize=True, reset_real_features=False).to(device)
    kid = KernelInceptionDistance(subset_size=50, normalize=True, reset_real_features=False).to(device)

    real_seen = 0
    for real_images, _ in tqdm(real_loader, desc="real features", leave=False):
        real_images = real_images.to(device)
        if real_seen >= num_fake:
            break
        current = min(real_images.size(0), num_fake - real_seen)
        real_batch = prepare_for_metrics(0.5 * (real_images[:current] + 1.0))
        fid.update(real_batch, real=True)
        kid.update(real_batch, real=True)
        real_seen += current

    fake_seen = 0
    while fake_seen < num_fake:
        batch_n = min(metric_batch_size, num_fake - fake_seen)
        fake_images = sample_fn(batch_n).to(device)
        fake_batch = prepare_for_metrics(fake_images)
        fid.update(fake_batch, real=False)
        kid.update(fake_batch, real=False)
        fake_seen += batch_n

    kid_mean, kid_std = kid.compute()
    return {
        "fid": float(fid.compute().item()),
        "kid_mean": float(kid_mean.item()),
        "kid_std": float(kid_std.item()),
    }


In [ ]:
sampler_metrics = {
    "ddpm": compute_sampler_metrics(lambda n: sample_ddpm(model, n_samples=n, show_progress=False), test_loader),
    "reverse_sde": compute_sampler_metrics(lambda n: sample_reverse_sde(model, n_samples=n, num_steps=300, show_progress=False), test_loader),
    "pf_ode": compute_sampler_metrics(lambda n: sample_probability_flow_ode(model, n_samples=n, num_steps=300, show_progress=False), test_loader),
}
sampler_metrics
